# Distributed MiniGPT Training & Serving on GKE Workspaces

### End-to-end ML Workflow: PySpark ETL, Switchable Multi-GPU / Multi-TPU Distributed Training, and 2-Replica GPU Inference

This example demonstrates orchestrating a complete machine learning lifecycle **directly from a lightweight CPU workspace** on Google Kubernetes Engine (GKE):

1. **Distributed Data Processing with Apache Spark**: Uses the Kubeflow Spark SDK to distribute WikiText tokenization and sequence chunking across Spark executors, writing compressed shards to Google Cloud Storage (GCS).
2. **Switchable Distributed PyTorch Training with Kubeflow Trainer**:
   - **Option A (GPU)**: 4x NVIDIA Tesla T4 GPUs via PyTorch DDP (`backend="nccl"`) on the `gpu-t4-spot` ComputeClass.
   - **Option B (TPU)**: 2x2 Cloud TPU v5e slice (4 chips) via TorchTPU (`backend="tpu_dist"`) on the `tpu-v5-4-single-host` ComputeClass.
3. **Cross-Hardware Model Weight Portability**: Validates saving model weights on Cloud TPU and seamlessly loading them into CUDA GPU memory.
4. **High-Availability Inference Serving**: Deploys a **2-replica GPU inference service** on NVIDIA Tesla T4 GPUs to serve real-time text generation.


## 0. Setup & Environment Verification

We load our Kubernetes and Kubeflow SDKs, verify in-cluster RBAC permissions, and configure our shared Google Cloud Storage (GCS) bucket.


In [ ]:
import os
import sys
import subprocess
import time

# Ensure jobs package is on sys.path
CURRENT_DIR = os.getcwd()
if os.path.isdir(os.path.join(CURRENT_DIR, "jobs")):
    JOBS_ROOT = CURRENT_DIR
elif os.path.isdir("/home/jovyan/minigpt/jobs"):
    JOBS_ROOT = "/home/jovyan/minigpt"
else:
    JOBS_ROOT = CURRENT_DIR

if JOBS_ROOT not in sys.path:
    sys.path.insert(0, JOBS_ROOT)

import kubeflow.spark
import kubeflow.trainer

NAMESPACE = os.environ.get("TENANT_NAMESPACE") or os.environ.get("DEMO_NAMESPACE")
if not NAMESPACE:
    ns_path = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    if os.path.exists(ns_path):
        try:
            with open(ns_path) as f:
                NAMESPACE = f.read().strip()
        except OSError:
            pass
    NAMESPACE = NAMESPACE or "kubeflow-user"

# Validate required GCS bucket environment variable
BUCKET_NAME = os.environ.get("GCS_BUCKET") or os.environ.get("DEMO_BUCKET")
if not BUCKET_NAME:
    raise ValueError(
        "Environment variable 'GCS_BUCKET' (or 'DEMO_BUCKET') is not set.\n"
        "Please configure your GCS bucket before running:\n"
        "    os.environ['GCS_BUCKET'] = '<your-project>-<your-namespace>-bucket'"
    )

# Container registry (defaults to prebuilt images on GHCR)
REGISTRY = os.environ.get("REGISTRY", "ghcr.io/gke-labs/workspaces").rstrip("/")

print(f"Kubernetes Namespace: {NAMESPACE}")
print(f"Shared GCS Bucket:    gs://{BUCKET_NAME}")
print(f"Artifact Registry:    {REGISTRY}")

# Verify notebook RBAC permissions for Spark, Trainer, and Deployments
for res in ["trainjobs.trainer.kubeflow.org", "sparkapplications.sparkoperator.k8s.io", "deployments.apps"]:
    out = subprocess.run(["kubectl", "auth", "can-i", "create", res, "-n", NAMESPACE],
                         capture_output=True, text=True)
    print(f"RBAC can-i create {res.split('.')[0]:20s}: {out.stdout.strip() or out.stderr.strip()}")


## Stage 1 — Distributed Data Processing with Apache Spark

We launch an Apache Spark cluster on Kubernetes using the `kubeflow.spark` SDK.
The Spark executors download raw WikiText data, tokenize it using the Hugging Face GPT-2 tokenizer, and chunk the token IDs into sequences of length `block_size + 1`.
Compressed `.npz` shards are written directly to `gs://{BUCKET_NAME}/processed/train/`.


In [ ]:
from jobs import pipeline

start_time = time.time()
spark_app = pipeline.run_data_processing(
    num_executors=2,
    num_shards=4,
    num_records=16000,
    block_size=128,
    wait=True,
)
print(f"\nSpark ETL processing completed in {time.time() - start_time:.2f} seconds.")


## Stage 2 — Switchable Distributed PyTorch Training

You can switch between two different distributed training configurations with a single parameter:
- **`"gpu"`**: 4x Tesla T4 GPUs across 4 distributed worker pods using PyTorch DDP with the `nccl` backend on the `gpu-t4-spot` ComputeClass.
- **`"tpu"`**: 2x2 Cloud TPU v5e slice (4 TPU chips on 1 host) using TorchTPU (`torch_tpu`) with the `tpu_dist` backend on the `tpu-v5-4-single-host` ComputeClass.

Both configurations use identical PyTorch `MiniGPT` model code and PyTorch `DistributedDataParallel` semantics.


In [ ]:
# Select your training hardware: "gpu" (4x Tesla T4) or "tpu" (2x2 TPU v5e slice)
ACCELERATOR = "tpu"  # Set to "gpu" or "tpu"

print(f"Launching distributed training targeting: {ACCELERATOR.upper()}")
start_train = time.time()

train_job_name = pipeline.run_training(
    accelerator=ACCELERATOR,
    epochs=2,
    batch_size=32,
    lr=3e-4,
    block_size=128,
    wait=True,
)
print(f"\nDistributed training finished in {time.time() - start_train:.2f} seconds.")


## Stage 3 — Inspect Model Checkpoints & Cross-Hardware Artifacts

When training completed, Rank 0 converted the model state dictionary to standard PyTorch CPU Float32 tensors (`{k: v.cpu().float()}`) and uploaded the weights to GCS (`gs://{BUCKET_NAME}/model/minigpt.pt`).

Let us inspect the training configuration and evaluation metrics emitted to GCS:


In [ ]:
from google.cloud import storage
import json

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

config_blob = bucket.get_blob("model/config.json")
metrics_blob = bucket.get_blob("model/metrics.json")
model_blob = bucket.get_blob("model/minigpt.pt")

print("=== Model Artifacts in GCS ===")
if model_blob:
    print(f"Weights:    gs://{BUCKET_NAME}/{model_blob.name} ({model_blob.size / 1024 / 1024:.2f} MB)")
if config_blob:
    print("\nConfig:")
    print(json.dumps(json.loads(config_blob.download_as_text()), indent=2))
if metrics_blob:
    print("\nMetrics:")
    print(json.dumps(json.loads(metrics_blob.download_as_text()), indent=2))


## Stage 4 — High-Availability GPU Inference Serving (2 Replicas on T4)

Now we answer the key question: **Is the model trained on Cloud TPU directly usable on NVIDIA GPUs?**

We deploy a Kubernetes Deployment with **2 replicas** on NVIDIA Tesla T4 GPUs (`nodeSelector: gpu-t4-spot`, `resources: {nvidia.com/gpu: 1}`).
Each replica loads `minigpt.pt` from GCS directly into CUDA memory on the T4 GPU and serves HTTP text generation requests.


In [ ]:
start_serve = time.time()
service_name = pipeline.deploy_inference(replicas=2, wait=True, timeout=600)
print(f"Inference deployment took {time.time() - start_serve:.2f} seconds.")


## Stage 5 — Querying the 2-Replica GPU Inference Cluster

We send test prompts across the 2-replica GPU inference service and observe high-throughput text generation powered by NVIDIA CUDA.


In [ ]:
from jobs import PROMPTS

pipeline.query_inference(service_name=service_name, prompts=PROMPTS, max_new_tokens=40)


## Cost & Resource Cleanup

When finished with the demo, tear down the inference service and temporary pods to let GKE automatically terminate the GPU and TPU node pools:

```bash
kubectl delete deployment minigpt-inference -n kubeflow-user
kubectl delete service minigpt-inference -n kubeflow-user
```
